In [1]:
!pip install arch neuralforecast pytorch-lightning


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.3/981.3 kB 4.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.1/309.1 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 831.6/831.6 kB 22.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.6/348.6 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 467.1/467.1 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 5.0 MB/s eta 0:00:00
  Attempting uninstall: tornado
    Found existing installation: tornado 6.5.1
    Uninstalling tornado-6.5.1:
      Successfully uninstalled tornado-6.5.1
  Attempting uninstall: pytorch-lightning
    Found existing installation: pytorch-lightning 2.6.5
    Uninstalling pytorch-lightning-2.6.5:
      Successfully uninstalled pytorch-lightning-2.6.5
ERROR: pip's dependency resolver does not currently take into account all

In [2]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob

# 1. LOAD DATA & MERGE (LOGIKA V1: LEFT JOIN + FFILL)
dl_path = "/kaggle/input/datasets/ferastoro/ai-stocks-crypto-and-ihsg/data/clean/dataset_deep_learning.csv"
tvp_path = "/kaggle/input/datasets/ferastoro/ai-stocks-crypto-and-ihsg/data/clean/tvp_var_dynamic_features.csv"

if not os.path.exists(dl_path):  
    dl_path = "data/clean/dataset_deep_learning.csv"
    tvp_path = "data/clean/tvp_var_dynamic_features.csv"

df_dl = pd.read_csv(dl_path)
df_tvp = pd.read_csv(tvp_path)
df_dl["Date"] = pd.to_datetime(df_dl["Date"])
df_tvp["Date"] = pd.to_datetime(df_tvp["Date"])

df = pd.merge(df_dl, df_tvp, on="Date", how="left")
tvp_cols = [c for c in df_tvp.columns if c != "Date"]
df[tvp_cols] = df[tvp_cols].ffill()
df.dropna(inplace=True)

# PENTING: JANGAN DROP INDEX, set "Date" sebagai index agar return bisa dilacak tanggalnya
df.set_index("Date", inplace=True)
print(f"Baris setelah merge TVP-VAR (harus 1676): {len(df)}")



Baris setelah merge TVP-VAR (harus 1676): 1676


In [3]:
# 2. NEURALFORECAST FORMAT & REAL_DATE
df["volatility_btc_idr"] = df["ret_crypto_btc_idr_indodax"]**2

nf_df = pd.DataFrame({
    "unique_id": "BTC_IDR",
    "ds": pd.date_range(start="2020-01-01", periods=len(df), freq="D"), 
    "y": df["volatility_btc_idr"].values
})

exog_cols = [
    "ret_saham_nvda", "ret_saham_msft", "ret_saham_tsla", "ret_saham_googl", "ret_saham_meta",
    "ret_crypto_btc_usd", "ret_crypto_eth_usd", "crypto_btc_usd_weekend_return",
    "spillover_ai_to_btcidr", "net_spill_saham_nvda", "net_spill_saham_tsla", "tvp_tsi",
    "gap_days", "is_monday", "is_ihsg_holiday"
]
for col in exog_cols:
    nf_df[col] = df[col].values

# MAPPING REAL DATE SEBELUM SPLIT (Ambil index Date yang sudah di-set)
nf_df["real_date"] = df.index.values

N = len(nf_df)
n_test = int(N * 0.20)
n_val = int(N * 0.10)
n_train = N - n_test - n_val
df_train = nf_df.iloc[:n_train].copy()
df_val = nf_df.iloc[n_train:n_train+n_val].copy()
df_test = nf_df.iloc[n_train+n_val:].copy()
print(f"Total data: {N} | Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}")



Total data: 1676 | Train: 1174 | Val: 167 | Test: 335


In [4]:
# ==============================================================
# BAGIAN A: PERBAIKAN EARLY STOPPING & RETRAIN 7 MODEL DL
# ==============================================================
from neuralforecast import NeuralForecast
from neuralforecast.models import TFT, LSTM, NHITS, TiDE, NBEATSx, TSMixerx, KAN
from pytorch_lightning.loggers import CSVLogger
import os

H = 10
INPUT_SIZE = 30
MAX_STEPS = 2000
VAL_CHECK_STEPS = 50
PATIENCE = 5
SEED = 42

models_list = [
    TFT(h=H, input_size=INPUT_SIZE, hist_exog_list=exog_cols, max_steps=MAX_STEPS, val_check_steps=VAL_CHECK_STEPS, early_stop_patience_steps=PATIENCE, random_seed=SEED, scaler_type="standard"),
    LSTM(h=H, input_size=INPUT_SIZE, hist_exog_list=exog_cols, max_steps=MAX_STEPS, val_check_steps=VAL_CHECK_STEPS, early_stop_patience_steps=PATIENCE, random_seed=SEED, scaler_type="standard"),
    NHITS(h=H, input_size=INPUT_SIZE, hist_exog_list=exog_cols, max_steps=MAX_STEPS, val_check_steps=VAL_CHECK_STEPS, early_stop_patience_steps=PATIENCE, random_seed=SEED, scaler_type="standard"),
    TiDE(h=H, input_size=INPUT_SIZE, hist_exog_list=exog_cols, max_steps=MAX_STEPS, val_check_steps=VAL_CHECK_STEPS, early_stop_patience_steps=PATIENCE, random_seed=SEED, scaler_type="standard"),
    NBEATSx(h=H, input_size=INPUT_SIZE, hist_exog_list=exog_cols, max_steps=MAX_STEPS, val_check_steps=VAL_CHECK_STEPS, early_stop_patience_steps=PATIENCE, random_seed=SEED, scaler_type="standard"),
    TSMixerx(h=H, input_size=INPUT_SIZE, hist_exog_list=exog_cols, n_series=1, max_steps=MAX_STEPS, val_check_steps=VAL_CHECK_STEPS, early_stop_patience_steps=PATIENCE, random_seed=SEED, scaler_type="standard"),
    KAN(h=H, input_size=INPUT_SIZE, hist_exog_list=exog_cols, max_steps=MAX_STEPS, val_check_steps=VAL_CHECK_STEPS, early_stop_patience_steps=PATIENCE, random_seed=SEED, scaler_type="standard")
]

print("Memulai training 7 Model DL (max_steps=2000, patience=5)...")
for m in models_list:
    m_name = m.__class__.__name__
    logger = CSVLogger(save_dir="/kaggle/working/logs", name=f"{m_name}_V3_Check")
    
    # PERBAIKAN 1: max_epochs = -1 (unlimited epochs, biar max_steps yang membatasi, bukan batas semu 1000)
    # PERBAIKAN 2: enable_checkpointing = True (wajib ada agar NeuralForecast merestore best weight di step awal)
    m.trainer_kwargs = {
        "logger": logger, 
        "enable_checkpointing": True, 
        "devices": 1,
        "max_epochs": 2000, "max_steps": 2000 
    }

nf = NeuralForecast(models=models_list, freq="D")
nf.fit(df=df_train.drop(columns=["real_date"]), val_size=n_val)

print("Training selesai. Memulai Cross-Validation...")
cv_df = nf.cross_validation(df=nf_df.drop(columns=["real_date"]), n_windows=n_test, step_size=1, val_size=n_val, refit=False)
cv_df = cv_df.reset_index(drop=True)

# MAPPING KEMBALI REAL_DATE KE HASIL CV
cv_df["ds"] = pd.to_datetime(cv_df["ds"])
nf_df_dates = nf_df[["unique_id", "ds", "real_date"]].copy()
nf_df_dates["ds"] = pd.to_datetime(nf_df_dates["ds"])
cv_df = pd.merge(cv_df, nf_df_dates, on=["unique_id", "ds"], how="left")

cv_df.to_csv("/kaggle/working/CV_Predictions_V3.csv", index=False)
print("Cross-Validation Selesai! Shape:", cv_df.shape)

Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores


Memulai training 7 Model DL (max_steps=2000, patience=5)...


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name                    | Type                     | Params | Mode 
-----------------------------------------------------------------------------
0 | loss                    | MAE                      | 0      | train
1 | hist_cat_embeddings     | ModuleList               | 0      | train
2 | futr_cat_embeddings     | ModuleList               | 0      | train
3 | stat_cat_embeddings     | ModuleList               | 0      | train
4 | padder_train            | ConstantPad1d            | 0      | train
5 | scaler                  | TemporalNorm             | 0      | train
6 | embedding               | TFTEmbedding             | 4.4 K  | train
7 | temporal_encoder        | TemporalCovariateEncoder | 1.9 M  | train
8 | temporal_fusion_decoder | TemporalFusionDecoder    | 256 K  | train
9 | output_adapter          | Linear                   | 129    | train
-----------------------------------------------------------------------------
2.2 M  

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Training: |          | 0/? [00:00<?, ?it/s]

... (474 outputs left out)

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_steps=2000` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting: |          | 0/? [00:00<?, ?it/s]

Cross-Validation Selesai! Shape: (3350, 12)


In [5]:
# CEK STATUS EARLY STOPPING DARI LOG
model_names = ["TFT", "LSTM", "NHITS", "TiDE", "NBEATSx", "TSMixerx", "KAN"]
stop_stats = []
for m in model_names:
    log_files = glob.glob(f"/kaggle/working/logs/{m}_V3_Check/*/metrics.csv")
    if log_files:
        m_df = pd.read_csv(log_files[-1])
        if "valid_loss" in m_df.columns:
            val_loss = m_df[["step", "valid_loss"]].dropna()
            if not val_loss.empty:
                min_val = val_loss["valid_loss"].min()
                min_step = val_loss.loc[val_loss["valid_loss"].idxmin(), "step"]
                last_step = val_loss["step"].iloc[-1]
                last_val = val_loss["valid_loss"].iloc[-1]
                status = "Early Stopped" if last_step < MAX_STEPS - VAL_CHECK_STEPS else "Max Steps Reached"
                stop_stats.append({
                    "Model": m, "Status": status, "Stopped_at_Step": last_step,
                    "Last_Val_Loss": last_val, "Min_Val_Loss": min_val, "Min_Val_Step": min_step
                })
df_stop = pd.DataFrame(stop_stats)
print("=== STATUS EARLY STOPPING V3 ===")
display(df_stop)



=== STATUS EARLY STOPPING V3 ===


,Model,Status,Stopped_at_Step,Last_Val_Loss,Min_Val_Loss,Min_Val_Step
0,TFT,Max Steps Reached,1999,0.000804,0.000746,99
1,LSTM,Max Steps Reached,1999,0.000705,0.000705,1999
2,NHITS,Max Steps Reached,1999,0.001375,0.001229,49
3,TiDE,Max Steps Reached,1999,0.000580,0.000579,1949
4,NBEATSx,Max Steps Reached,1999,0.001437,0.001227,49
5,TSMixerx,Max Steps Reached,1999,0.000876,0.000744,149
6,KAN,Max Steps Reached,1999,0.001360,0.000950,49


In [6]:
# ==============================================================
# BAGIAN B: BASELINE GARCH(1,1) DAN EGARCH (ROLLING FORECAST)
# ==============================================================
from arch import arch_model
import time

returns_pct = df["ret_crypto_btc_idr_indodax"] * 100
train_returns = returns_pct.iloc[:n_train+n_val]
test_returns = returns_pct.iloc[n_train+n_val:]

print("=== FITTING MODEL GARCH (TRAIN+VAL) ===")
garch11 = arch_model(train_returns, vol="GARCH", p=1, q=1, dist="normal")
print(garch11.fit(disp="off").summary())

print("\n=== FITTING MODEL EGARCH (TRAIN+VAL) ===")
egarch = arch_model(train_returns, vol="EGARCH", p=1, q=1, dist="normal")
print(egarch.fit(disp="off", options={"maxiter": 1000}).summary())

garch_h1, egarch_h1 = [], []
garch_h10, egarch_h10 = [], []

print(f"\nMemulai Rolling Forecast (Total iterasi: {len(test_returns)})...")
start_time = time.time()

for i in range(len(test_returns)):
    window = returns_pct.iloc[:n_train+n_val+i]
    m_garch = arch_model(window, vol="GARCH", p=1, q=1, dist="normal").fit(disp="off", show_warning=False)
    m_egarch = arch_model(window, vol="EGARCH", p=1, q=1, dist="normal").fit(disp="off", show_warning=False, options={"maxiter": 1000})
    
    f_garch = m_garch.forecast(horizon=10, reindex=False)
    
    # PERBAIKAN: EGARCH h>1 di arch TIDAK support method="analytic", sehingga butuh simulasi (default 1000).
    f_egarch = m_egarch.forecast(horizon=10, reindex=False, method="simulation", simulations=1000)
    
    garch_h1.append(f_garch.variance.values[-1, 0] / (100**2))
    egarch_h1.append(f_egarch.variance.values[-1, 0] / (100**2))
    garch_h10.append(f_garch.variance.values[-1, 9] / (100**2))
    egarch_h10.append(f_egarch.variance.values[-1, 9] / (100**2))
    
    if (i + 1) % 50 == 0:
        elapsed = time.time() - start_time
        est_total = (elapsed / (i + 1)) * len(test_returns)
        print(f"Iterasi {i+1}/{len(test_returns)} selesai. Estimasi sisa waktu: {est_total - elapsed:.1f} detik")

# PERBAIKAN (Metodologi Laporan): 
# Memotong / Capping varians simulasi EGARCH ke ambang batas wajar (Persentil 99.9% dari actual variance Train)
# Agar model lain tidak terkompromi/drop N jika simulasi EGARCH meledak sampai tak terhingga.
var_cap = np.percentile(df_train["y"].values, 99.9)
egarch_h1 = np.clip(egarch_h1, 0, var_cap)
egarch_h10 = np.clip(egarch_h10, 0, var_cap)
print(f"\nCapping limit numerik EGARCH Variance (99.9% Train): {var_cap}")

df_garch = pd.DataFrame({
    "real_date": test_returns.index,
    "GARCH11_h1": garch_h1, "EGARCH_h1": egarch_h1,
    "GARCH11_h10": garch_h10, "EGARCH_h10": egarch_h10
})

=== FITTING MODEL GARCH (TRAIN+VAL) ===
                         Constant Mean - GARCH Model Results                          
Dep. Variable:     ret_crypto_btc_idr_indodax   R-squared:                       0.000
Mean Model:                     Constant Mean   Adj. R-squared:                  0.000
Vol Model:                              GARCH   Log-Likelihood:               -3497.26
Distribution:                          Normal   AIC:                           7002.52
Method:                    Maximum Likelihood   BIC:                           7023.33
                                                No. Observations:                 1341
Date:                        Wed, Sep 30 2026   Df Residuals:                     1340
Time:                                15:36:46   Df Model:                            1
                                 Mean Model                                
                 coef    std err          t      P>|t|     95.0% Conf. Int.
-------------------------

/usr/local/lib/python3.12/dist-packages/arch/univariate/volatility.py:2838: RuntimeWarning: overflow encountered in exp
  paths[loc, :, :] = np.exp(_lnsigma2[:, m:])
/usr/local/lib/python3.12/dist-packages/arch/univariate/volatility.py:2838: RuntimeWarning: overflow encountered in exp
  paths[loc, :, :] = np.exp(_lnsigma2[:, m:])


Iterasi 50/335 selesai. Estimasi sisa waktu: 96.2 detik


/usr/local/lib/python3.12/dist-packages/arch/univariate/volatility.py:2838: RuntimeWarning: overflow encountered in exp
  paths[loc, :, :] = np.exp(_lnsigma2[:, m:])


Iterasi 100/335 selesai. Estimasi sisa waktu: 65.6 detik


/usr/local/lib/python3.12/dist-packages/arch/univariate/volatility.py:2838: RuntimeWarning: overflow encountered in exp
  paths[loc, :, :] = np.exp(_lnsigma2[:, m:])


Iterasi 150/335 selesai. Estimasi sisa waktu: 52.2 detik
Iterasi 200/335 selesai. Estimasi sisa waktu: 37.1 detik
Iterasi 250/335 selesai. Estimasi sisa waktu: 19.6 detik
Iterasi 300/335 selesai. Estimasi sisa waktu: 7.1 detik

Capping limit numerik EGARCH Variance (99.9% Train): 0.032611637736022964


In [7]:
# ==============================================================
# BAGIAN C: EVALUASI GABUNGAN
# ==============================================================
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from scipy.stats import norm

df_h1 = cv_df.groupby("cutoff").nth(0).reset_index()
df_h10 = cv_df.groupby("cutoff").nth(9).reset_index()

df_h1 = pd.merge(df_h1, df_garch[["real_date", "GARCH11_h1", "EGARCH_h1"]], on="real_date", how="left")
df_h1.rename(columns={"GARCH11_h1": "GARCH11", "EGARCH_h1": "EGARCH"}, inplace=True)
df_h10 = pd.merge(df_h10, df_garch[["real_date", "GARCH11_h10", "EGARCH_h10"]], on="real_date", how="left")
df_h10.rename(columns={"GARCH11_h10": "GARCH11", "EGARCH_h10": "EGARCH"}, inplace=True)

full_y_by_date = nf_df.set_index("ds")["y"].sort_index()

def build_persistence(df):
    prev_dates = df["ds"] - pd.Timedelta(days=1)
    y_persist = prev_dates.map(full_y_by_date)
    y_persist = y_persist.fillna(df["y"]) 
    return y_persist.values

df_h1["y_persist"] = build_persistence(df_h1)
df_h10["y_persist"] = build_persistence(df_h10)
df_h1["y_hist_mean"] = df_train["y"].mean()
df_h10["y_hist_mean"] = df_train["y"].mean()

# Ganti Inf menjadi NaN
df_h1 = df_h1.replace([np.inf, -np.inf], np.nan)
df_h10 = df_h10.replace([np.inf, -np.inf], np.nan)

# FUNGSI EVALUASI METRIK (PERBAIKAN MASKING INDEPENDEN)
def evaluate_models_consistently(df_subset):
    all_models = ["TFT", "LSTM", "NHITS", "TiDE", "NBEATSx", "TSMixerx", "KAN", "GARCH11", "EGARCH"]
    results = []
    
    for m in all_models:
        # MASKING HANYA UNTUK MODEL "m" - N tiap model bisa beda-beda sekarang
        mask = df_subset["y"].notna() & df_subset[m].notna()
        df_eval = df_subset[mask].copy()
        
        y_true = df_eval["y"].values
        y_pred = df_eval[m].values
        
        N_eval = len(y_true)
        if N_eval < 2: continue
        
        r2 = r2_score(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        mae = mean_absolute_error(y_true, y_pred)
        
        t_diff = np.diff(y_true)
        p_diff = y_pred[1:] - y_true[:-1]
        dir_acc = np.mean(np.sign(t_diff) == np.sign(p_diff)) * 100 if len(t_diff) > 0 else np.nan
        
        results.append({"Model": m, "N": N_eval, "R2": r2, "RMSE": rmse, "MAE": mae, "DirAcc (%)": dir_acc})
    
    return pd.DataFrame(results)

res_h1 = evaluate_models_consistently(df_h1)
res_h10 = evaluate_models_consistently(df_h10)

print("=== METRIK h=1 (short horizon) ===")
display(res_h1.round(6))
print("\n=== METRIK h=10 (long horizon) ===")
display(res_h10.round(6))

# DIEBOLD MARIANO TEST
def diebold_mariano_test_nw(y_true, y_pred_model, y_pred_baseline, lag=0):
    T = len(y_true)
    if T < max(3, lag + 2): return T, 0.0, 1.0

    d = (y_true - y_pred_baseline)**2 - (y_true - y_pred_model)**2
    d_mean = np.mean(d)

    gamma0 = np.var(d, ddof=0)
    nw_var = gamma0
    for k in range(1, lag + 1):
        gamma_k = np.cov(d[:-k], d[k:])[0, 1]
        weight = 1 - k / (lag + 1)
        nw_var += 2 * weight * gamma_k

    if nw_var <= 0: return T, 0.0, 1.0
    DM_stat = d_mean / np.sqrt(nw_var / T)
    p_value = 2 * (1 - norm.cdf(abs(DM_stat)))
    return T, DM_stat, p_value

def run_dm_for_horizon(df_subset, lag, title):
    dm_results = []
    alpha_bonf = 0.05 / 18
    all_models = ["TFT", "LSTM", "NHITS", "TiDE", "NBEATSx", "TSMixerx", "KAN", "GARCH11", "EGARCH"]
    
    for m in all_models:
        # DM Test VS PERSISTENCE
        mask_p = df_subset["y"].notna() & df_subset[m].notna() & df_subset["y_persist"].notna()
        y_t_p = df_subset.loc[mask_p, "y"].values
        y_m_p = df_subset.loc[mask_p, m].values
        y_b_p = df_subset.loc[mask_p, "y_persist"].values
        N_p, dm_stat_p, p_val_p = diebold_mariano_test_nw(y_t_p, y_m_p, y_b_p, lag=lag)
        
        dm_results.append({
            "Model": m, "Baseline": "Persistence", "N": N_p,
            "DM Statistic": dm_stat_p, "p-value mentah": p_val_p,
            "Signifikan (Bonferroni)?": bool(p_val_p < alpha_bonf)
        })
        
        # DM Test VS HISTORICAL MEAN
        mask_hm = df_subset["y"].notna() & df_subset[m].notna() & df_subset["y_hist_mean"].notna()
        y_t_hm = df_subset.loc[mask_hm, "y"].values
        y_m_hm = df_subset.loc[mask_hm, m].values
        y_b_hm = df_subset.loc[mask_hm, "y_hist_mean"].values
        N_hm, dm_stat_hm, p_val_hm = diebold_mariano_test_nw(y_t_hm, y_m_hm, y_b_hm, lag=lag)
        
        dm_results.append({
            "Model": m, "Baseline": "Historical Mean", "N": N_hm,
            "DM Statistic": dm_stat_hm, "p-value mentah": p_val_hm,
            "Signifikan (Bonferroni)?": bool(p_val_hm < alpha_bonf)
        })
        
    print(f"\n=== {title} ===")
    display(pd.DataFrame(dm_results))

run_dm_for_horizon(df_h1, lag=0, title="DM TEST h=1 (lag=0)")
run_dm_for_horizon(df_h10, lag=9, title="DM TEST h=10 (lag=9)")

=== METRIK h=1 (short horizon) ===


,Model,N,R2,RMSE,MAE,DirAcc (%)
0,TFT,335,-0.311866,0.001763,0.000737,68.862275
1,LSTM,335,-0.415719,0.001831,0.000614,72.455090
2,NHITS,335,-0.823768,0.002079,0.000890,58.982036
3,TiDE,335,-0.064666,0.001588,0.000482,70.958084
4,NBEATSx,335,-6.896870,0.004325,0.001018,60.479042
5,TSMixerx,335,-0.305928,0.001759,0.000676,70.059880
6,KAN,335,-4.564935,0.003631,0.001250,61.676647
7,GARCH11,326,-0.076623,0.001617,0.000750,65.230769
8,EGARCH,326,-62.049417,0.012378,0.005991,58.769231



=== METRIK h=10 (long horizon) ===


,Model,N,R2,RMSE,MAE,DirAcc (%)
0,TFT,335,-1.025040,0.002192,0.000724,68.562874
1,LSTM,335,-0.500448,0.001887,0.000615,70.059880
2,NHITS,335,-0.802930,0.002068,0.000918,64.371257
3,TiDE,335,-0.051764,0.001580,0.000485,72.754491
4,NBEATSx,335,-1.955855,0.002648,0.000977,66.167665
5,TSMixerx,335,-0.880707,0.002113,0.000806,67.065868
6,KAN,335,-2.769566,0.002991,0.001128,61.077844
7,GARCH11,335,-0.108339,0.001622,0.000813,64.371257
8,EGARCH,331,-58.943335,0.011994,0.005692,58.484848



=== DM TEST h=1 (lag=0) ===


,Model,Baseline,N,DM Statistic,p-value mentah,Signifikan (Bonferroni)?
0,TFT,Persistence,335,-0.334987,7.376351e-01,False
1,TFT,Historical Mean,335,-0.853882,3.931702e-01,False
2,LSTM,Persistence,335,-0.501679,6.158932e-01,False
3,LSTM,Historical Mean,335,-0.666986,5.047812e-01,False
4,NHITS,Persistence,335,-2.916466,3.540216e-03,False
5,NHITS,Historical Mean,335,-1.984856,4.716045e-02,False
6,TiDE,Persistence,335,0.491543,6.230428e-01,False
7,TiDE,Historical Mean,335,1.821773,6.848939e-02,False
8,NBEATSx,Persistence,335,-1.040335,2.981843e-01,False
9,NBEATSx,Historical Mean,335,-1.041223,2.977719e-01,False



=== DM TEST h=10 (lag=9) ===


,Model,Baseline,N,DM Statistic,p-value mentah,Signifikan (Bonferroni)?
0,TFT,Persistence,335,-1.006005,0.314413,False
1,TFT,Historical Mean,335,-1.016617,0.309336,False
2,LSTM,Persistence,335,-0.777196,0.437043,False
3,LSTM,Historical Mean,335,-0.710100,0.477642,False
4,NHITS,Persistence,335,-1.998183,0.045697,False
5,NHITS,Historical Mean,335,-1.818019,0.069061,False
6,TiDE,Persistence,335,1.447831,0.147664,False
7,TiDE,Historical Mean,335,1.460532,0.144144,False
8,NBEATSx,Persistence,335,-1.286665,0.198211,False
9,NBEATSx,Historical Mean,335,-1.255475,0.209306,False


In [8]:
# PERBANDINGAN R2 V2 LAMA vs V3 BARU
v2_r2_h1 = {"TiDE": -0.058658, "NBEATSx": -3.832755, "TSMixerx": -0.087203, "KAN": -0.646542, "TFT": -0.301822, "LSTM": -0.402284, "NHITS": -1.433591}
v2_r2_h10 = {"TiDE": -0.054176, "NBEATSx": -2.563219, "TSMixerx": -0.071753, "KAN": -0.826003, "TFT": -0.246750, "LSTM": -0.403717, "NHITS": -1.369400}

comp_data = []
for m in ["TFT", "LSTM", "NHITS", "TiDE", "NBEATSx", "TSMixerx", "KAN"]:
    r2_v3_h1 = res_h1.loc[res_h1["Model"] == m, "R2"].values[0]
    r2_v3_h10 = res_h10.loc[res_h10["Model"] == m, "R2"].values[0]
    
    stop_stat = df_stop.loc[df_stop["Model"] == m]
    if not stop_stat.empty:
        status = f"{stop_stat["Status"].values[0]} (Step {stop_stat["Stopped_at_Step"].values[0]})"
        min_loss = stop_stat["Min_Val_Loss"].values[0]
    else:
        status, min_loss = "Unknown", np.nan
        
    comp_data.append({
        "Model": m, "Early Stop Status": status, "Min Val Loss": min_loss,
        "R² h=1 (V2 No EarlyStop)": v2_r2_h1.get(m), "R² h=1 (V3 Fixed)": r2_v3_h1, "Selisih h=1": r2_v3_h1 - v2_r2_h1.get(m),
        "R² h=10 (V2 No EarlyStop)": v2_r2_h10.get(m), "R² h=10 (V3 Fixed)": r2_v3_h10, "Selisih h=10": r2_v3_h10 - v2_r2_h10.get(m)
    })

print("=== PERBANDINGAN PERFORMA AKIBAT EARLY STOPPING FIX ===")
display(pd.DataFrame(comp_data).round(6))



=== PERBANDINGAN PERFORMA AKIBAT EARLY STOPPING FIX ===


,Model,Early Stop Status,Min Val Loss,R² h=1 (V2 No EarlyStop),R² h=1 (V3 Fixed),Selisih h=1,R² h=10 (V2 No EarlyStop),R² h=10 (V3 Fixed),Selisih h=10
0,TFT,Max Steps Reached (Step 1999),0.000746,-0.301822,-0.311866,-0.010044,-0.246750,-1.025040,-0.778290
1,LSTM,Max Steps Reached (Step 1999),0.000705,-0.402284,-0.415719,-0.013435,-0.403717,-0.500448,-0.096731
2,NHITS,Max Steps Reached (Step 1999),0.001229,-1.433591,-0.823768,0.609823,-1.369400,-0.802930,0.566470
3,TiDE,Max Steps Reached (Step 1999),0.000579,-0.058658,-0.064666,-0.006008,-0.054176,-0.051764,0.002412
4,NBEATSx,Max Steps Reached (Step 1999),0.001227,-3.832755,-6.896870,-3.064115,-2.563219,-1.955855,0.607364
5,TSMixerx,Max Steps Reached (Step 1999),0.000744,-0.087203,-0.305928,-0.218725,-0.071753,-0.880707,-0.808954
6,KAN,Max Steps Reached (Step 1999),0.000950,-0.646542,-4.564935,-3.918393,-0.826003,-2.769566,-1.943563
